# Non LLM flow
START -> Temerature conversion -> Label tempeature -> END

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict,Literal

class Temperature(TypedDict):
    temperature_cent : float
    temperature_farn : float
    label : Literal["HEAT","COLD","WARM"]

#create graph
graph = StateGraph(state_schema=Temperature)

#create node
def tempConverter(state :Temperature )->Temperature:
    celciusTemperature = state["temperature_cent"]
    state["temperature_farn"]=(celciusTemperature * 9 / 5) + 32
    return state

def defineLabel(state :Temperature): 
    if state["temperature_cent"] > 25:
        state["label"] = "HEAT"
    elif state["temperature_cent"] > 10:
        state["label"] = "WARM"
    else:
        state["label"] = "COLD"
    return state

# add nodes to graph
graph.add_node("temperatureConversion",tempConverter)
graph.add_node("temperatureLabel",defineLabel)

# add edges
graph.add_edge(START, "temperatureConversion")
graph.add_edge("temperatureConversion","temperatureLabel")
graph.add_edge("temperatureLabel",END)

# Compile the graph
workflow = graph.compile()

# invoke graph
initial_state = {"temperature_cent":28.5}
final_state = workflow.invoke(initial_state)
print(final_state)




In [ ]:
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())


# LLM flow

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="openai/gpt-oss-20b")

In [ ]:
from langgraph.graph import StateGraph, START,END
from typing import TypedDict

#get state
class QuestionAns(TypedDict):
    question : str
    answer : str

#node
def processQuestion(state : QuestionAns):
    question = state["question"]
    state["answer"] = llm.invoke(question).content
    return state

#add node
graph = StateGraph(state_schema=QuestionAns)
graph.add_node("QuestionAnswer",processQuestion)

#add edges
graph.add_edge(START, "QuestionAnswer")
graph.add_edge("QuestionAnswer",END)

workflow = graph.compile()

#invoke
initial_state = {"question":"How is weather in bangalore?"}
final_state = workflow.invoke(initial_state)

print(final_state)


In [ ]:
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())

# prompt chaining

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class BlogState(TypedDict):
    title : str
    create_outline : str
    create_blog : str

#create nodes
def createOutline(state : BlogState):
    prompt = f"Create ONLY required outline for title : {state['title']}"
    state["create_outline"]= llm.invoke(prompt).content
    return state

def createBlog(state : BlogState):
    prompt = f" Create content for title {state['title']} and given outline for title : {state['create_outline']}"
    state["create_blog"]= llm.invoke(prompt).content
    return state


#add nodes
graph = StateGraph(state_schema=BlogState)
graph.add_node("createOutline",createOutline)
graph.add_node("createBlog",createBlog)

#add edges 
graph.add_edge(START,"createOutline")
graph.add_edge("createOutline","createBlog")
graph.add_edge("createBlog",END)

workflow = graph.compile()

initial_state = {"title":"How to crack AI interviews"}
final_state = workflow.invoke(initial_state)

print(final_state["create_blog"])


In [ ]:
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())